In [ ]:
import os
import json
import glob
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Environment ready.")

In [ ]:
import os

print(os.listdir('/kaggle/input'))

In [ ]:
import os

dataset_path = "/kaggle/input/datasets"

print(os.listdir(dataset_path))

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/jannatulferdaues"

print(os.listdir(dataset_path))

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/jannatulferdaues/ghid-professional-dataset"

print(os.listdir(dataset_path))

In [ ]:
DATA_PATH = "/kaggle/input/datasets/jannatulferdaues/ghid-professional-dataset/professional_profiles_dataset.csv"

df = pd.read_excel(DATA_PATH)

# Drop the stray "Unnamed: 0" index column that Excel exports sometimes carry
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Shape:", df.shape)
df.head()

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

# The model will learn:

User Profile
     
     ↓
     
Education

Major

Desired Role

Language Level

Wanted Education

Health

Stress

Earnings

Chosen Country  
     
     ↓ 
     
ML Model 
     
     ↓
     
Best Fit Country  
    
     ↓
     
Top-5 Country Recommendations

In [ ]:
# ============================================================
# TARGET + INPUT FEATURES
# ============================================================

TARGET = "best_fit_country"

FEATURES = [
    "education",
    "major",
    "desired_role",
    "language_level",
    "wanted_education",
    "health",
    "stress",
    "earnings_usd",
    "chosen_country"
]

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("Features:")
print(FEATURES)

print("\nTarget:")
print(TARGET)

print("\nNumber of target classes:", y.nunique())

display(y.value_counts().to_frame("count"))

In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

def preprocess_features(data):
    data = data[FEATURES].copy()

    # Convert earnings into a less-skewed representation
    data["earnings_log1p"] = np.log1p(data["earnings_usd"])

    # Normalize stress from 0-10 to 0-1
    data["stress_normalized"] = data["stress"] / 10.0

    return data


X_processed = preprocess_features(X)

print("Processed shape:", X_processed.shape)

display(X_processed.head())

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

In [ ]:
# ============================================================
# CATBOOST MODEL
# ============================================================

from catboost import CatBoostClassifier

categorical_features = [
    "education",
    "major",
    "desired_role",
    "language_level",
    "wanted_education",
    "health",
    "chosen_country"
]

cat_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]

print("Categorical feature indices:", cat_indices)

In [ ]:
# ============================================================
# CLASS WEIGHTS
# ============================================================

class_counts = y_train.value_counts()

n_classes = len(class_counts)
total_samples = len(y_train)

class_weights = {}

for country, count in class_counts.items():
    weight = total_samples / (n_classes * count)

    # Prevent extremely rare classes from dominating training
    class_weights[country] = min(weight, 5.0)

print("Class weights:")
print(class_weights)